# Question Data from Metaculus API (v3 - Score Fix)

**Date:** 2026-02-11  
**Version:** 010b - Fixed score data extraction  
**Input:** `products/Run_Question_Map_2026-02-10_v01.csv`  
**Output:** `products/Question_Data_from_API_2026-02-11_vNN.csv`

**Fix in this version:**
- ✅ Scores now extracted from correct path: `question.aggregations.unweighted.score_data`
- ✅ Community forecast from `question.aggregations.unweighted.latest`

**Note:** These are COMMUNITY scores (how well the community predicted). Bot's personal scores require authentication.

In [ ]:
# Imports
import requests
import pandas as pd
import time
import json
from pathlib import Path
from datetime import date
from typing import Dict, Optional

print("✅ Imports successful")

In [ ]:
# Configuration
INPUT_FILE = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products/Run_Question_Map_2026-02-10_v01.csv")
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
API_BASE = "https://www.metaculus.com/api2/questions"
TEST_LIMIT = None  # Set to 5 for testing, None for all questions

# Rate limiting settings
RATE_LIMIT_DELAY = 2.0  # Seconds between requests
MAX_RETRIES = 3  # Max retry attempts on 429 error
BACKOFF_BASE = 5  # Base delay for exponential backoff (seconds)
PROGRESS_SAVE_INTERVAL = 25  # Save progress every N questions

print(f"Input file: {INPUT_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Test limit: {TEST_LIMIT}")
print(f"Rate limit: {RATE_LIMIT_DELAY}s between requests")
print(f"Max retries: {MAX_RETRIES}")
print(f"Progress saves: every {PROGRESS_SAVE_INTERVAL} questions")

In [ ]:
# Load question list
df_runs = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_runs)} run records")

# Extract unique question numbers (filter out empty values)
question_numbers = df_runs['question_number'].dropna().astype(int).unique().tolist()
question_numbers.sort()

print(f"Found {len(question_numbers)} unique questions")
print(f"Range: {min(question_numbers)} to {max(question_numbers)}")
print(f"First 10: {question_numbers[:10]}")

In [ ]:
# API fetch function with retry logic
def fetch_question_data(question_id: int, max_retries: int = MAX_RETRIES) -> Optional[Dict]:
    """Fetch question data from Metaculus API with exponential backoff retry."""
    url = f"{API_BASE}/{question_id}/"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            
            # Success
            if response.status_code == 200:
                return response.json()
            
            # Rate limit - retry with exponential backoff
            if response.status_code == 429:
                wait_time = BACKOFF_BASE * (2 ** attempt)
                print(f"\n  ⏳ Rate limited, waiting {wait_time}s... ", end='')
                time.sleep(wait_time)
                continue
            
            # Other error
            response.raise_for_status()
            
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                print(f"\n  ⚠️  API error after {max_retries} attempts: {e}")
                return None
            # Wait before retry
            time.sleep(2)
    
    return None

print("✅ fetch_question_data() with retry logic defined")

In [ ]:
# Data extraction function - FIXED SCORE PATH
def extract_question_fields(data: Dict) -> Dict:
    """Extract relevant fields from API response."""
    
    # Top-level fields
    result = {
        'question_id': data.get('id'),
        'title': data.get('title', ''),
        'short_title': data.get('short_title', ''),
        'slug': data.get('slug', ''),
        'status': data.get('status', ''),
        'resolved': data.get('resolved', False),
        'comment_count': data.get('comment_count', 0),
        'nr_forecasters': data.get('nr_forecasters', 0),
        'forecasts_count': data.get('forecasts_count', 0),
        'author_username': data.get('author_username', ''),
        'curation_status': data.get('curation_status', ''),
    }
    
    # Dates
    result['created_at'] = data.get('created_at', '')
    result['published_at'] = data.get('published_at', '')
    result['edited_at'] = data.get('edited_at', '')
    result['open_time'] = data.get('open_time', '')
    result['actual_close_time'] = data.get('actual_close_time', '')
    result['scheduled_close_time'] = data.get('scheduled_close_time', '')
    result['actual_resolve_time'] = data.get('actual_resolve_time', '')
    result['scheduled_resolve_time'] = data.get('scheduled_resolve_time', '')
    
    # Question sub-object
    question = data.get('question', {})
    result['question_type'] = question.get('type', '')
    result['resolution'] = question.get('resolution')
    result['resolution_set_time'] = question.get('resolution_set_time', '')
    result['question_weight'] = question.get('question_weight', '')
    result['description'] = question.get('description', '')
    result['resolution_criteria'] = question.get('resolution_criteria', '')
    result['fine_print'] = question.get('fine_print', '')
    
    # Question-type specific fields
    if result['question_type'] == 'multiple_choice':
        result['mc_options'] = json.dumps(question.get('options', []))
    else:
        result['mc_options'] = ''
    
    if result['question_type'] == 'numeric':
        scaling = question.get('scaling', {})
        result['numeric_range_min'] = scaling.get('range_min', '')
        result['numeric_range_max'] = scaling.get('range_max', '')
        result['open_upper_bound'] = scaling.get('open_upper_bound', '')
        result['open_lower_bound'] = scaling.get('open_lower_bound', '')
    else:
        result['numeric_range_min'] = ''
        result['numeric_range_max'] = ''
        result['open_upper_bound'] = ''
        result['open_lower_bound'] = ''
    
    # Tournament info
    projects = data.get('projects', {})
    default_project = projects.get('default_project', {})
    result['tournament_id'] = default_project.get('id', '')
    result['tournament_name'] = default_project.get('name', '')
    result['tournament_slug'] = default_project.get('slug', '')
    
    # FIXED: Get aggregations from question.aggregations (not top-level aggregations)
    question_aggregations = question.get('aggregations', {})
    unweighted = question_aggregations.get('unweighted', {})
    latest = unweighted.get('latest', {})
    
    # Community forecast from latest
    result['community_forecaster_count'] = latest.get('forecaster_count', '')
    forecast_values = latest.get('forecast_values', [])
    
    # Format community forecast based on type
    if result['question_type'] == 'binary' and len(forecast_values) == 2:
        result['community_forecast'] = f"{forecast_values[1]:.1%}"  # p_yes
        result['community_forecast_mean'] = forecast_values[1]
    elif result['question_type'] == 'multiple_choice':
        result['community_forecast'] = json.dumps(forecast_values)
        result['community_forecast_mean'] = ''
    elif result['question_type'] == 'numeric':
        means = latest.get('means', [])
        if means:
            result['community_forecast_mean'] = means[0]
            result['community_forecast'] = f"{means[0]:.2f}"
        else:
            result['community_forecast'] = ''
            result['community_forecast_mean'] = ''
    else:
        result['community_forecast'] = ''
        result['community_forecast_mean'] = ''
    
    # Interval bounds from latest
    interval_lower = latest.get('interval_lower_bounds', [])
    interval_upper = latest.get('interval_upper_bounds', [])
    result['community_interval_lower'] = interval_lower[0] if interval_lower else ''
    result['community_interval_upper'] = interval_upper[0] if interval_upper else ''
    
    # FIXED: Score data from question.aggregations.unweighted.score_data (not latest.score_data)
    score_data = unweighted.get('score_data', {})
    result['coverage'] = score_data.get('coverage', '')
    result['peer_score'] = score_data.get('peer_score', '')
    result['baseline_score'] = score_data.get('baseline_score', '')
    result['spot_peer_score'] = score_data.get('spot_peer_score', '')
    result['spot_baseline_score'] = score_data.get('spot_baseline_score', '')
    
    return result

print("✅ extract_question_fields() defined (SCORE PATH FIXED)")

In [ ]:
# Helper: Find next version number for output file
def get_next_version_file(base_name: str) -> Path:
    """Get next available version number for output file."""
    version = 1
    while True:
        filename = OUTPUT_DIR / f"{base_name}_v{version:02d}.csv"
        if not filename.exists():
            return filename
        version += 1

# Helper: Check if question already fetched (for resume capability)
def load_existing_results(base_name: str) -> pd.DataFrame:
    """Load most recent output file if it exists."""
    import glob
    pattern = str(OUTPUT_DIR / f"{base_name}_v*.csv")
    files = sorted(glob.glob(pattern))
    if files:
        print(f"Found existing file: {Path(files[-1]).name}")
        return pd.read_csv(files[-1])
    return pd.DataFrame()

print("✅ Helper functions defined")

In [ ]:
# Test on single question
test_id = question_numbers[0]
print(f"Testing API fetch for question {test_id}...")

test_data = fetch_question_data(test_id)
if test_data:
    print(f"✅ API response received ({len(test_data)} top-level keys)")
    test_fields = extract_question_fields(test_data)
    print(f"✅ Extracted {len(test_fields)} fields")
    print("\nSample fields:")
    for k in ['question_id', 'title', 'question_type', 'status', 'resolved', 'resolution']:
        print(f"  {k}: {test_fields.get(k)}")
    print("\nScore fields:")
    for k in ['coverage', 'peer_score', 'baseline_score', 'spot_peer_score', 'spot_baseline_score']:
        print(f"  {k}: {test_fields.get(k)}")
else:
    print("❌ API fetch failed")

In [ ]:
# Batch fetch with improved rate limiting and progress saves
questions_to_fetch = question_numbers[:TEST_LIMIT] if TEST_LIMIT else question_numbers

# Check for existing results (resume capability)
base_name = f"Question_Data_from_API_{date.today()}"
existing_df = load_existing_results(base_name)
already_fetched = set(existing_df['question_id'].tolist()) if not existing_df.empty else set()

if already_fetched:
    print(f"\nResuming: {len(already_fetched)} questions already fetched")
    questions_to_fetch = [q for q in questions_to_fetch if q not in already_fetched]
    print(f"Remaining: {len(questions_to_fetch)} questions\n")
else:
    print(f"\nFetching {len(questions_to_fetch)} questions...\n")

results = existing_df.to_dict('records') if not existing_df.empty else []
failed = []
last_save_count = len(results)

for i, qnum in enumerate(questions_to_fetch, 1):
    print(f"[{i}/{len(questions_to_fetch)}] Q{qnum}...", end='')
    
    data = fetch_question_data(qnum)
    if data:
        try:
            fields = extract_question_fields(data)
            results.append(fields)
            print(f" ✓ ({fields['question_type']}, {fields['status']})")
        except Exception as e:
            print(f" ⚠️  Extract failed: {e}")
            failed.append(qnum)
    else:
        print(f" ❌ Fetch failed")
        failed.append(qnum)
    
    # Periodic progress save
    if (len(results) - last_save_count) >= PROGRESS_SAVE_INTERVAL:
        temp_df = pd.DataFrame(results)
        progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
        temp_df.to_csv(progress_file, index=False)
        print(f"  💾 Progress saved: {len(results)} questions")
        last_save_count = len(results)
    
    # Rate limiting (be respectful)
    if i < len(questions_to_fetch):
        time.sleep(RATE_LIMIT_DELAY)

print(f"\n✅ Successfully fetched {len(results)} total questions")
if failed:
    print(f"❌ Failed to fetch {len(failed)} questions: {failed}")

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")

In [ ]:
# Data quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

print(f"\nQuestion Types:")
print(df['question_type'].value_counts())

print(f"\nStatus:")
print(df['status'].value_counts())

print(f"\nResolved:")
print(df['resolved'].value_counts())

print(f"\nResolution completeness (for resolved questions):")
resolved_df = df[df['resolved'] == True]
print(f"  Total resolved: {len(resolved_df)}")
print(f"  With resolution value: {resolved_df['resolution'].notna().sum()}")
print(f"  Missing resolution: {resolved_df['resolution'].isna().sum()}")

print(f"\nForecaster counts:")
print(f"  With nr_forecasters > 0: {(df['nr_forecasters'] > 0).sum()}")
print(f"  With community_forecaster_count > 0: {(df['community_forecaster_count'] != '').sum()}")

print(f"\nComment counts:")
print(f"  With comment_count > 0: {(df['comment_count'] > 0).sum()}")

print(f"\nCoverage/Score data (FIXED):")
print(f"  With coverage: {(df['coverage'] != '').sum()}")
print(f"  With peer_score: {(df['peer_score'] != '').sum()}")
print(f"  With baseline_score: {(df['baseline_score'] != '').sum()}")
print(f"  With spot_peer_score: {(df['spot_peer_score'] != '').sum()}")
print(f"  With spot_baseline_score: {(df['spot_baseline_score'] != '').sum()}")

print(f"\nTournaments:")
print(df['tournament_name'].value_counts())

In [ ]:
# Save to VERSIONED CSV
output_file = get_next_version_file(base_name)
df.to_csv(output_file, index=False)
print(f"\n✅ Saved to: {output_file.name}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")

# Clean up progress file if it exists
progress_file = OUTPUT_DIR / f"{base_name}_progress.csv"
if progress_file.exists():
    progress_file.unlink()
    print(f"   Cleaned up progress file")

In [ ]:
# Display sample of key fields including scores
print("\n" + "=" * 80)
print("SAMPLE DATA (first 10 questions)")
print("=" * 80)

key_cols = ['question_id', 'short_title', 'question_type', 'status', 'resolved', 
            'resolution', 'coverage', 'peer_score', 'baseline_score']
available_cols = [c for c in key_cols if c in df.columns]
df[available_cols].head(10)